# Computational Theory Problems

This notebook works through a series of problems based on the Secure Hash Standard published by NIST as FIPS PUB 180-4. 

## Problem 1: Representing SHA-256 Data

SHA-256 works on fixed-size binary data. Before implementing the algorithm, this section chooses a Python representation for each kind of data it uses, based on the definitions in FIPS 180-4.

The choice matters because Python's built-in `int` has no fixed size. SHA-256 depends on values that are exactly 32 bits wide and that wrap around when they overflow ([FIPS 180-4, Section 3.2](https://doi.org/10.6028/NIST.FIPS.180-4)). The wrong representation produces a wrong hash without raising any error.

For each kind of data, this section:

- states what the standard requires
- explains why the chosen representation is appropiate
- demonstrates the chosen representation with an example

All multi-byte values are read and written in big-endian order (see Section 4).



| Data | Size | Representation |
| --- | --- | --- |
| Word | 32 bits | `np.uint32` |
| Sequence of words | $n \times 32$ bits | 1-D NumPy array of `np.uint32` |
| Input message | Any whole number of bytes | `bytes` |
| Message block | 512 bits | NumPy array of 16 `np.uint32` words |
| Hash value | 256 bits | NumPy array of 8 `np.uint32` words, output as a 64-character hex string |


Each choice is explained and demonstrated below.

### Imports

[NumPy](https://numpy.org/doc/stable/user/basics.types.html) provides fixed-width integer types such as `uint32`. The standard library module `sys` is used later to check the byte order of the machine ([`sys.byteorder`](https://docs.python.org/3/library/sys.html#sys.byteorder)). The standard library module [`hashlib`](https://docs.python.org/3/library/hashlib.html) provides Python's own SHA-256, which is used to check results. order.

In [1]:
# Standard library: Python's built-in SHA-256, used to check results.
import hashlib

# Standard library: reports this machine's native byte order.
import sys

# NumPy: provides fixed-width unsigned integers such as uint32.
import numpy as np

### 1. 32-bit Words

FIPS 180-4 ([Section 2.1](https://doi.org/10.6028/NIST.FIPS.180-4)) defines a *word* as a group of $w$ bits, with $w = 32$ for SHA-256. Every SHA-256 operation works on words.

**Choice:** NumPy's `np.uint32`, an unsigned (non-negative) integer that is always exactly 32 bits wide.

**Why it is appropriate:**

- It can't hold more than 32 bits which matches the defintion of a word
- Addition wraps around after $2^{32} - 1$, as required by Section 3.2
- A lift shift removes any bit moved after the 32nd position

Each of these properties is checked below.


In [2]:
# Create a single 32-bit word holding the largest possible value.
word = np.uint32(0xFFFFFFFF)

print(word)
print(type(word))
print(hex(word))

4294967295
<class 'numpy.uint32'>
0xffffffff


**Exactly 32 bits.**

 A 32-bit unsigned integer ranges from $0$ to $2^{32} - 1$. Instead of entering these limits by hand, [`np.iinfo`](https://numpy.org/doc/stable/reference/generated/numpy.iinfo.html) reads them directly from NumPy.

In [3]:
# Show the smallest and largest values a uint32 can hold.
limits = np.iinfo(np.uint32)

print(limits.min, limits.max)
print(hex(limits.max))

0 4294967295
0xffffffff


**Addition wraps modulo $2^{32}$.** 

Adding 1 to the largest `uint32` value gives 0. The same sum with a Python `int` is included for contrast: it continues to 4294967296, which is not a valid 32-bit word.

An array is used here rather than a single value. Adding two single `np.uint32` values also gives 0, but NumPy prints an overflow warning as well. An array wraps without the warning.

In [4]:
# A uint32 array wraps round to 0 when it overflows.
words = np.array([0xFFFFFFFF], dtype=np.uint32)
print(words + np.uint32(1))

# A Python int keeps growing past 32 bits.
print(0xFFFFFFFF + 1)

[0]
4294967296


**Shifting stays within 32 bits.** 

The value `0x80000001` has its highest and lowest bits set. Shifting it left by one moves the highest bit outside the 32-bit range. A `uint32` removes that bit and gives `0x2`, while a Python `int` keeps it and becomes 33 bits long.

In [5]:
# A value with its highest and lowest bits set.
x = np.uint32(0x80000001)

# Shift left by one: the highest bit falls off a uint32.
print(hex(x << 1))

# The same shift on a Python int keeps the extra bit.
print(hex(0x80000001 << 1))

0x2
0x100000002


### 2. Sequences of 32-bit Words

SHA-256 uses several fixed-length sequences of words

8 words for the hash value ([FIPS 180-4, Section 5.3.3](https://doi.org/10.6028/NIST.FIPS.180-4)), 
16 for each message block (Section 5.2.1), and 64 each for the round constants (Section 4.2.2) and the message schedule (Section 6.2.2).

**Choice:** a one-dimensional [NumPy array](https://numpy.org/doc/stable/reference/arrays.ndarray.html) with `dtype=np.uint32`.

**Why it is appropriate:**

- Every element is stored as a `uint32`, so each one keeps the 32-bit behaviour from Section 1
- An operation applied to the whole array is applied to every element at once and each result still wraps at 32 bits
- The words are kept in order and can be accessed by position which matches the subscript notation in the standard like $H_0$ to $H_7$

**Fixed type and size.** 

The eight initial hash values $H^{(0)}$ are stored below. The array reports its type, its length and its total size in bytes. Eight words of 4 bytes each give 32 bytes which is 256 bits.

In [6]:
# The eight initial hash values H(0) from FIPS 180-4, Section 5.3.3.
initial_hash = np.array(
    [0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
     0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19],
    dtype=np.uint32,
)

print(initial_hash.dtype, initial_hash.shape, initial_hash.nbytes)

# A single element keeps the uint32 type.
print(type(initial_hash[0]))

uint32 (8,) 32
<class 'numpy.uint32'>


**Operations stay within 32 bits.**

 Here the array is added to itself so every value gets doubled in one go. Each answer wraps around at $2^{32}$ so they all stay as proper 32-bit words. The same values in a normal Python `list` are shown underneath for comparason. Four of them go over 32 bits, which you can see from the extra `1` at the front.

In [7]:
# Every element is added at once, and each sum wraps at 2**32.
print([hex(word) for word in initial_hash + initial_hash])

# The same values in a Python list are plain ints and do not wrap.
hash_list = [0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
             0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19]
print([hex(word + word) for word in hash_list])

['0xd413ccce', '0x76cf5d0a', '0x78dde6e4', '0x4a9fea74', '0xa21ca4fe', '0x360ad118', '0x3f07b356', '0xb7c19a32']
['0xd413ccce', '0x176cf5d0a', '0x78dde6e4', '0x14a9fea74', '0xa21ca4fe', '0x1360ad118', '0x3f07b356', '0xb7c19a32']


### 3. Input Messages

The standard treats the message as a sequence of bits with a length of $\ell$ bits ([FIPS 180-4, Section 5.1.1](https://doi.org/10.6028/NIST.FIPS.180-4)). In practice, every message is made up of whole bytes, so $\ell$ is always a multiple of 8.

**Choice:** Python's built-in [`bytes`](https://docs.python.org/3/library/stdtypes.html#bytes-objects) type.

**Why it is appropriate:**

- Each element is one byte, a whole number from 0 to 255, so the message is stored exactly as SHA-256 reads it.
- The length in bits, $\ell$, is simply `len(message) * 8`. This value is needed for padding in Problem 4.
- A `bytes` object cannot be changed after it is created, so the message cannot be altered by mistake while it is being hashed.
- Python's [`hashlib.sha256`](https://docs.python.org/3/library/hashlib.html) also takes `bytes`, so the same input can be passed to both when testing in Problem 6.

**Text must be encoded first.** 

SHA-256 works on bytes, not text, so a string is converted with [`str.encode`](https://docs.python.org/3/library/stdtypes.html#str.encode) using UTF-8 (the standard way of storing text as bytes). The message `"abc"` is used here because it is one of the standard SHA-256 test messages published by NIST.

In [8]:
# Convert the text "abc" into bytes using UTF-8.
message = "abc".encode("utf-8")

print(message)
print(type(message))

# Each byte is a number from 0 to 255.
print(list(message))

# The message length in bits, written as l in the standard.
print(len(message) * 8, "bits")

b'abc'
<class 'bytes'>
[97, 98, 99]
24 bits


**Characters and bytes are not the same.** 

In UTF-8, a basic English letter takes 1 byte, but other characters can take 2 to 4 bytes. This means the message length has to come from the encoded bytes, not from the number of characters in the text. This matters for the non-ASCII test messages in Problem 6.

In [9]:
# Compare the number of characters with the number of bytes after encoding.
for text in ["abc", "café", "€"]:
    encoded = text.encode("utf-8")
    print(f"{text!r} -> characters: {len(text)}, bytes: {len(encoded)}, {encoded}")

'abc' -> characters: 3, bytes: 3, b'abc'
'café' -> characters: 4, bytes: 5, b'caf\xc3\xa9'
'€' -> characters: 1, bytes: 3, b'\xe2\x82\xac'


**The message cannot be changed by mistake.** 

`bytes` objects are immutable (cannot be changed after they are created). Trying to change a byte raises an error instead of quietly altering the message.

In [10]:
# Try to replace the first byte; bytes objects do not allow this.
try:
    message[0] = 120
except TypeError as error:
    print("TypeError:", error)

TypeError: 'bytes' object does not support item assignment


### 4. Big-Endian Integers

When a number is stored in more than one byte, the bytes can be written in two orders. *Big-endian* puts the most significant byte (the "biggest" part of the number) first, the same way numbers are written on paper. *Little-endian* puts the least significant byte first. The terms come from Danny Cohen's 1980 note [*On Holy Wars and a Plea for Peace*](https://www.rfc-editor.org/ien/ien137.txt).

The Secure Hash Standard uses the big-endian convention throughout ([FIPS 180-4, Section 3.1](https://doi.org/10.6028/NIST.FIPS.180-4)). This affects two later steps: reading each group of 4 message bytes as one word (Problem 4), and writing the message length into the padding (Problem 4).

**Choice:** always state the byte order explicitly. Python integers use [`int.to_bytes` and `int.from_bytes`](https://docs.python.org/3/library/stdtypes.html#int.to_bytes) with `"big"`. NumPy arrays use the dtype `">u4"`, where `>` means big-endian, `u` means unsigned and `4` means 4 bytes ([NumPy byte order](https://numpy.org/doc/stable/user/basics.byteswapping.html)).

**Why it is appropriate:**

- Most modern processors are little-endian, so leaving the order to the machine gives the bytes in the wrong order for SHA-256.
- Naming the order makes the code give the same result on every machine.
- No error is raised when the order is wrong, so it has to be correct by design rather than caught later.

**This machine's byte order.** 

[`sys.byteorder`](https://docs.python.org/3/library/sys.html#sys.byteorder) reports the order the machine uses by default.

In [11]:
# Show the native byte order of the machine running this notebook.
print(sys.byteorder)

little


**One number, two byte orders.** 

The number `0x12345678` is made of the four bytes `12`, `34`, `56` and `78`. Big-endian keeps them in the order they are written. Little-endian reverses them.

In [12]:
# Write the same 32-bit number as 4 bytes in each order.
value = 0x12345678

# hex(" ") puts a space between each byte so the order is easy to see.
print("big:   ", value.to_bytes(4, "big").hex(" "))
print("little:", value.to_bytes(4, "little").hex(" "))

big:    12 34 56 78
little: 78 56 34 12


**Reading bytes with the wrong order.** 

The same four bytes give two different numbers depending on the order used to read them. NumPy's plain `np.uint32` uses the machine's own order, which on a little-endian machine gives the wrong word for SHA-256. The `">u4"` dtype forces big-endian and gives the correct word on any machine.

In [13]:
# Four bytes as they would appear in a message.
raw = bytes([0x12, 0x34, 0x56, 0x78])

# Python integers: the order is named in the call.
print("int, big:      ", hex(int.from_bytes(raw, "big")))
print("int, little:   ", hex(int.from_bytes(raw, "little")))

# NumPy: ">u4" is big-endian; plain np.uint32 uses the machine's order.
print("numpy, >u4:    ", hex(np.frombuffer(raw, dtype=">u4")[0]))
print("numpy, native: ", hex(np.frombuffer(raw, dtype=np.uint32)[0]))

int, big:       0x12345678
int, little:    0x78563412
numpy, >u4:     0x12345678
numpy, native:  0x78563412


**The message length in the padding.**

 SHA-256 ends the padded message with the original message length in bits, written as a 64-bit (8-byte) big-endian integer ([FIPS 180-4, Section 5.1.1](https://doi.org/10.6028/NIST.FIPS.180-4)). For the 3-byte message `"abc"`, the length is 24 bits, which is `0x18` in hexadecimal.

In [14]:
# The length of "abc" in bits, as an 8-byte big-endian integer.
message_length = len(b"abc") * 8
print(message_length.to_bytes(8, "big").hex(" "))

00 00 00 00 00 00 00 18


### 5. 512-bit Message Blocks

After padding, the message is split into blocks of 512 bits ([FIPS 180-4, Section 5.2.1](https://doi.org/10.6028/NIST.FIPS.180-4)). Each block is then read as sixteen 32-bit words, written $M_0^{(i)}$ to $M_{15}^{(i)}$ for block $i$. These sixteen words become the first sixteen entries of the message schedule in the compression function (Section 6.2.2).

$$512 \text{ bits} = 64 \text{ bytes} = 16 \text{ words} \times 32 \text{ bits}$$

**Choice:** a NumPy array of 16 `np.uint32` words, made from 64 bytes with [`np.frombuffer`](https://numpy.org/doc/stable/reference/generated/numpy.frombuffer.html) using the big-endian dtype `">u4"`, then converted with `.astype(np.uint32)`.

**Why it is appropriate:**

- `np.frombuffer` turns the bytes into words in one step, without a Python loop.
- `">u4"` reads each group of 4 bytes big-endian, as required by Section 3.1 (see Section 4 above).
- `.astype(np.uint32)` converts the words into the machine's own `uint32` type, so they behave the same as every other word in the notebook.
- The result is the same kind of array as in Section 2, so the functions in later problems can use it directly.

**Building one block.** 

The padded form of `"abc"` fits in a single block: the 3 message bytes, a `0x80` byte (a `1` bit followed by seven `0` bits), 52 zero bytes, and the message length of 24 bits as an 8-byte big-endian integer. The padding rules are implemented properly in Problem 4. Here the block is built by hand to give a real example.

In [15]:
# The padded "abc" message: message, 0x80, zeros, then the length in bits.
padded = b"abc" + b"\x80" + bytes(52) + (24).to_bytes(8, "big")

print(len(padded), "bytes,", len(padded) * 8, "bits")

64 bytes, 512 bits


**Reading the block as 16 words.** 

`np.frombuffer` with `">u4"` gives 16 big-endian words. The `.astype(np.uint32)` step converts them to the machine's own `uint32` type. It also makes a copy that can be changed, since an array made by `np.frombuffer` is read-only.

In [16]:
# Read each group of 4 bytes as one big-endian 32-bit word.
raw_words = np.frombuffer(padded, dtype=">u4")
print(raw_words.dtype, raw_words.shape, "writeable:", raw_words.flags.writeable)

# Convert to the machine's own uint32 type, as used everywhere else.
block = raw_words.astype(np.uint32)
print(block.dtype, block.shape, "writeable:", block.flags.writeable)

>u4 (16,) writeable: False
uint32 (16,) writeable: True


**Checking the words.** 

The first word holds `"abc"` and the `0x80` byte, and the last word holds the length, 24 (`0x18`). These match the `"abc"` example published by NIST on its [example values page](https://csrc.nist.gov/projects/cryptographic-standards-and-guidelines/example-values).

In [17]:
# Print each word as 8 hex digits, labelled M0 to M15 as in the standard.
for index, word in enumerate(block):
    print(f"M{index:<2} = {int(word):08x}")

M0  = 61626380
M1  = 00000000
M2  = 00000000
M3  = 00000000
M4  = 00000000
M5  = 00000000
M6  = 00000000
M7  = 00000000
M8  = 00000000
M9  = 00000000
M10 = 00000000
M11 = 00000000
M12 = 00000000
M13 = 00000000
M14 = 00000000
M15 = 00000018


**Longer messages give more blocks.** 

A padded message is always a multiple of 64 bytes. [`reshape(-1, 16)`](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html) arranges the words into rows of 16, one row per block, where `-1` tells NumPy to work out the number of rows itself. This example uses 128 bytes with the values 0 to 127, which gives two blocks.

In [18]:
# 128 example bytes (0x00 to 0x7F): enough for exactly two blocks.
two_blocks = np.frombuffer(bytes(range(128)), dtype=">u4").astype(np.uint32)

# One row of 16 words per block.
two_blocks = two_blocks.reshape(-1, 16)
print(two_blocks.shape)

# The first word of each block.
print(f"{int(two_blocks[0][0]):08x} {int(two_blocks[1][0]):08x}")

(2, 16)
00010203 40414243


### 6. The Final 256-bit Hash Value

The hash value is made of eight 32-bit words, $H_0$ to $H_7$, giving $8 \times 32 = 256$ bits ([FIPS 180-4, Section 6.2.2](https://doi.org/10.6028/NIST.FIPS.180-4)). After the last block is processed, these eight words are joined together to give the final message digest (the finished hash).

**Choice:** while the algorithm runs, the hash value is a NumPy array of 8 `np.uint32` words. The final output is a 64-character lowercase hexadecimal string.

**Why it is appropriate:**

- An array of 8 `uint32` words is the same kind of sequence as in Section 2, so updating the hash after each block wraps at 32 bits as required.
- Each word is written as exactly 8 hex digits, and $8 \times 8 = 64$ characters, which is the usual way a SHA-256 hash is displayed.
- Python's [`hashlib.sha256(...).hexdigest()`](https://docs.python.org/3/library/hashlib.html#hashlib.hash.hexdigest) returns the same format, so results can be compared directly as strings in Problem 6.

**Eight words, 256 bits.** 

The eight words below are the SHA-256 hash of `"abc"`, taken from the NIST [example values page](https://csrc.nist.gov/projects/cryptographic-standards-and-guidelines/example-values). Calculating them is the job of Problems 5 and 6.

In [19]:
# The SHA-256 hash of "abc" as eight 32-bit words, from NIST's example.
final_hash = np.array(
    [0xBA7816BF, 0x8F01CFEA, 0x414140DE, 0x5DAE2223,
     0xB00361A3, 0x96177A9C, 0xB410FF61, 0xF20015AD],
    dtype=np.uint32,
)

print(final_hash.dtype, final_hash.shape, final_hash.nbytes * 8, "bits")

uint32 (8,) 256 bits


**Converting to a hex string.** 

Each word is formatted as 8 lowercase hex digits with `08x`, and the eight pieces are joined into one string. The result matches `hashlib.sha256(b"abc").hexdigest()` exactly.

In [20]:
# Format each word as 8 hex digits and join them into one string.
digest = "".join(f"{int(word):08x}" for word in final_hash)
print(digest)
print(len(digest), "characters")

# Compare with Python's built-in SHA-256.
expected = hashlib.sha256(b"abc").hexdigest()
print(digest == expected)

ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad
64 characters
True


**Keeping the leading zeros.** Python's 

`hex()` drops leading zeros, so a word such as `0x0000ABCD` would become only 4 characters and the digest would be too short. The format `08x` always gives 8 characters, padding with zeros on the left.

In [21]:
# A word whose first four hex digits are zero.
word = np.uint32(0x0000ABCD)

# hex() drops the leading zeros (the [2:] removes the "0x" prefix).
print(hex(word)[2:], len(hex(word)[2:]), "characters")

# 08x keeps them, so every word is exactly 8 characters.
print(f"{int(word):08x}", len(f"{int(word):08x}"), "characters")

abcd 4 characters
0000abcd 8 characters


**The digest as raw bytes.**

 A hash can also be given as 32 raw bytes, which is what [`hashlib.sha256(...).digest()`](https://docs.python.org/3/library/hashlib.html#hashlib.hash.digest) returns. Converting the words with `">u4"` writes each one big-endian, giving the same bytes. The hex string is used as the main output because the brief asks for it and because it is readable.

In [22]:
# Write each word as 4 big-endian bytes: 8 words give 32 bytes.
digest_bytes = final_hash.astype(">u4").tobytes()
print(len(digest_bytes), "bytes")

# The bytes match hashlib's raw digest, and give the same hex string.
print(digest_bytes == hashlib.sha256(b"abc").digest())
print(digest_bytes.hex() == digest)

32 bytes
True
True


## Problem 2: SHA-256 Bitwise Operations

SHA-256 mixes the bits of its working data using six functions, defined in [FIPS 180-4, Section 4.1.2](https://doi.org/10.6028/NIST.FIPS.180-4). Each function takes one or three 32-bit words and returns one 32-bit word. They are built only from the bitwise operations in Section 2.2.2 of the standard:

| Symbol | Operation | Python |
| --- | --- | --- |
| $\land$ | AND: 1 only where both bits are 1 | `&` |
| $\oplus$ | XOR: 1 where the bits differ | `^` |
| $\lnot$ | NOT: flips every bit | `~` |
| $\text{SHR}^n$ | shift right by $n$ places, filling with zeros | `>>` |
| $\text{ROTR}^n$ | rotate right by $n$ places, wrapping bits round to the left | `>>`, `<<` and `|` combined |

The six functions are:

| Function | Definition |
| --- | --- |
| $Ch(x, y, z)$ | $(x \land y) \oplus (\lnot x \land z)$ |
| $Maj(x, y, z)$ | $(x \land y) \oplus (x \land z) \oplus (y \land z)$ |
| $\Sigma_0^{256}(x)$ | $\text{ROTR}^{2}(x) \oplus \text{ROTR}^{13}(x) \oplus \text{ROTR}^{22}(x)$ |
| $\Sigma_1^{256}(x)$ | $\text{ROTR}^{6}(x) \oplus \text{ROTR}^{11}(x) \oplus \text{ROTR}^{25}(x)$ |
| $\sigma_0^{256}(x)$ | $\text{ROTR}^{7}(x) \oplus \text{ROTR}^{18}(x) \oplus \text{SHR}^{3}(x)$ |
| $\sigma_1^{256}(x)$ | $\text{ROTR}^{17}(x) \oplus \text{ROTR}^{19}(x) \oplus \text{SHR}^{10}(x)$ |

The four $\Sigma$ and $\sigma$ functions share one pattern: three values XORed together. The capital $\Sigma$ functions use three rotations, while the lower-case $\sigma$ functions replace the last rotation with a shift. Only the amounts differ.

Every function works on `np.uint32` values, as chosen in Problem 1, so all results stay within 32 bits.

### Helper Operations: SHR and ROTR

The $\Sigma$ and $\sigma$ functions are built from two operations defined in [FIPS 180-4, Section 3.2](https://doi.org/10.6028/NIST.FIPS.180-4), for $0 \le n < 32$:

$$\text{SHR}^n(x) = x \gg n$$

$$\text{ROTR}^n(x) = (x \gg n) \lor (x \ll (32 - n))$$

A **shift** moves every bit $n$ places to the right. The $n$ bits that fall off the right-hand end are lost, and zeros come in on the left.

A **rotation** also moves every bit $n$ places to the right, but the bits that fall off the right-hand end come back in on the left. No bits are lost. In the formula, $x \gg n$ gives the bits that moved right, $x \ll (32 - n)$ moves the fallen-off bits to the left-hand end, and $\lor$ (OR) combines the two.

The left shift only works because `np.uint32` drops any bit pushed past bit 32, as shown in Problem 1, Section 1. With a plain Python `int`, the extra bits would remain and the rotation would be wrong.

In [23]:
# The number of bits in a SHA-256 word, used by the rotation below.
WORD_BITS = 32

**SHR: shift right.** 

The input is converted to `np.uint32` first, so the function also works if a plain Python `int` is passed in. The shift amount is converted too, so both sides of `>>` are the same type.

In [24]:
def shr(x, n):
    """
    Shift a 32-bit word right by n places (SHR in FIPS 180-4, Section 3.2).

    Bits shifted off the right-hand end are lost, and zeros fill in
    from the left.

    Parameters
    ----------
    x : int or np.uint32 or np.ndarray
        The word (or array of words) to shift.
    n : int
        The number of places to shift, where 0 <= n < 32.

    Returns
    -------
    np.uint32 or np.ndarray
        The shifted word (or words).
    """
    x = np.uint32(x)
    return x >> np.uint32(n)

**Testing SHR.**

 Each test uses [`assert`](https://docs.python.org/3/reference/simple_stmts.html#the-assert-statement), which raises an error if its condition is false and does nothing if it is true. The inputs are chosen so the correct answer can be checked by hand: shifting `0x80000000` (only the top bit set) right by 31 leaves only the bottom bit, and shifting `0xFFFFFFFF` right by 4 replaces the top four bits with zeros.

In [25]:
# Only the top bit set: shifting right by 31 moves it to the bottom.
assert shr(0x80000000, 31) == 0x00000001

# All bits set: shifting right by 4 brings in four zeros on the left.
assert shr(0xFFFFFFFF, 4) == 0x0FFFFFFF

# The lowest bit falls off the end and is lost.
assert shr(0x00000001, 1) == 0x00000000

# The result is still a 32-bit word.
assert isinstance(shr(0x12345678, 3), np.uint32)

print("All shr tests passed.")

All shr tests passed.


**ROTR: rotate right.** 

The code follows the formula directly: shift right by $n$, shift left by $32 - n$, and combine with OR.

In [26]:
def rotr(x, n):
    """
    Rotate a 32-bit word right by n places (ROTR in FIPS 180-4, Section 3.2).

    Bits rotated off the right-hand end come back in on the left, so no
    bits are lost.

    Parameters
    ----------
    x : int or np.uint32 or np.ndarray
        The word (or array of words) to rotate.
    n : int
        The number of places to rotate, where 0 < n < 32.

    Returns
    -------
    np.uint32 or np.ndarray
        The rotated word (or words).
    """
    x = np.uint32(x)

    # The bits that move right, combined with the bits that wrap round.
    return (x >> np.uint32(n)) | (x << np.uint32(WORD_BITS - n))

**Testing ROTR.** 

The first two tests show bits wrapping round: the lowest bit rotated right by 1 becomes the highest bit, and rotating `0x12345678` right by 8 moves the last byte, `78`, to the front. The third test uses the fact that rotating right by 13 and then by 19 is a full turn of 32 places, which must give back the original word. The last test shows the function also works on a whole array at once.

In [27]:
# The lowest bit wraps round to become the highest bit.
assert rotr(0x00000001, 1) == 0x80000000

# Rotating by 8 bits moves the last byte to the front.
assert rotr(0x12345678, 8) == 0x78123456

# 13 + 19 = 32 places is a full turn, giving back the original word.
assert rotr(rotr(0x12345678, 13), 19) == 0x12345678

# Works on every element of a uint32 array at once.
words = np.array([0x00000001, 0x80000000], dtype=np.uint32)
assert np.array_equal(rotr(words, 1), [0x80000000, 0x40000000])

print("All rotr tests passed.")

All rotr tests passed.


### 1. Ch: Choose

$$Ch(x, y, z) = (x \land y) \oplus (\lnot x \land z)$$

Defined in [FIPS 180-4, Section 4.1.2](https://doi.org/10.6028/NIST.FIPS.180-4). For each bit position, the bit of $x$ chooses between $y$ and $z$: where $x$ is 1 the result takes the bit from $y$, and where $x$ is 0 it takes the bit from $z$.

The formula does this in two halves:

- $x \land y$ keeps the bits of $y$ where $x$ is 1, and gives 0 everywhere else.
- $\lnot x \land z$ keeps the bits of $z$ where $x$ is 0, and gives 0 everywhere else.

The two halves never have a 1 in the same position, so combining them with XOR ($\oplus$) simply joins them together.

| $x$ | $y$ | $z$ | $Ch$ |
| :-: | :-: | :-: | :-: |
| 0 | 0 | 0 | 0 |
| 0 | 0 | 1 | 1 |
| 0 | 1 | 0 | 0 |
| 0 | 1 | 1 | 1 |
| 1 | 0 | 0 | 0 |
| 1 | 0 | 1 | 0 |
| 1 | 1 | 0 | 1 |
| 1 | 1 | 1 | 1 |

In the top half of the table $x$ is 0, so $Ch$ copies the $z$ column. In the bottom half $x$ is 1, so $Ch$ copies the $y$ column.

**Why NOT needs `np.uint32`.** 

Python's `~` on a plain `int` does not give a 32-bit result. It gives a negative number, because a Python `int` has no fixed width. On an `np.uint32`, `~` flips exactly 32 bits, as the standard requires.

In [28]:
# NOT on a plain Python int gives a negative number.
print(~0x0000FFFF)

# NOT on a uint32 flips exactly 32 bits.
print(hex(~np.uint32(0x0000FFFF)))

-65536
0xffff0000


In [29]:
def Ch(x, y, z):
    """
    Choose bits from y or z, using x (Ch in FIPS 180-4, Section 4.1.2).

    For each bit position, the result takes the bit from y where x is 1
    and the bit from z where x is 0.

    Parameters
    ----------
    x : int or np.uint32 or np.ndarray
        The word whose bits do the choosing.
    y : int or np.uint32 or np.ndarray
        The word chosen from where x is 1.
    z : int or np.uint32 or np.ndarray
        The word chosen from where x is 0.

    Returns
    -------
    np.uint32 or np.ndarray
        The chosen bits as a 32-bit word (or words).
    """
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)

    # Bits of y where x is 1, joined with bits of z where x is 0.
    return (x & y) ^ (~x & z)

**Testing Ch.** 

When $x$ is all 1s, every bit should come from $y$. When $x$ is all 0s, every bit should come from $z$. When $x$ is `0xFFFF0000`, the top half should come from $y$ and the bottom half from $z$. These cases can each be checked by eye.


In [30]:
y = 0x12345678
z = 0x9ABCDEF0

# x all 1s: every bit comes from y.
assert Ch(0xFFFFFFFF, y, z) == y

# x all 0s: every bit comes from z.
assert Ch(0x00000000, y, z) == z

# x is 1s on the left and 0s on the right: left half from y, right half from z.
assert Ch(0xFFFF0000, y, z) == 0x1234DEF0

# The result is still a 32-bit word.
assert isinstance(Ch(1, 2, 3), np.uint32)

print("All Ch tests passed.")

All Ch tests passed.
